Complete codes to processing CoastCams

# 📖 Table of Contents
- [Get camera intrinsics and extrinsics](#Get-camera-intrinsics-and-extrinsics)
- [Define timestacks for ARGUS camera](#Define-timestacks-for-ARGUS-camera)
- [Rectify image products](#Rectify-image-products)
    - [Example: Plot image rectification from netcdf](##Example-plot-of-rectified-image)
    - [Example: Rectify single image](##Rectifying-single-image)
- [Extract runup](#Extract-runup)
    - [Example: Runup from single image](##Runup-from-single-image)
    - [Codes for DEM plots](##DEM-plots)
- [Extract shorelines](#Extract-shorelines)
    - [Example: Shoreline from single image](##Shoreline-from-single-image)


## Get camera intrinsics and extrinsics

In [ ]:

import pandas as pd
import os
if 'CODES' not in sys.path:
    sys.path.append('CODES')

import utils_CIRN
configPath = "config.json"
print(os.getcwd())
sample_img_path = 'DATA/1726950600.Sat.Sep.21_20_30_00.GMT.2024.CACO03.c2.snap.jpg'

#pix_filename = "filename for pulling .pix file - end in .pix"
pix_filename = 'DATA/CACO03_c2_timestack_20240827.pix'

img_handle = ImageHandler(imagePath = sample_img_path, configPath = configPath)

print(f'sample image path= {sample_img_path}')


img_handle.read_metadata()
metadata = img_handle.metadata
config = img_handle.config

productsPath = config.get("productsPath", {})
pixsaveDir = config.get("pixsaveDir", {})
      
df = pd.read_csv(pix_filename, 
                         sep='\s+',  # or sep=r"\s+"
                         header=None,            # No header in file
                         names=["Ud", "Vd"])  # Assign column names

xyz = utils_CIRN.dist_uv_to_xyz(metadata['intrinsics'], 
                         metadata['extrinsics'],  df["Ud"], df["Vd"], 'z', 0, timeout = 60)

print(xyz)
xyzdf=pd.DataFrame(xyz)
xyzdf.to_csv('transect_11072024_c2_xyz.csv')

## Define timestacks for ARGUS camera

In [ ]:
import sys
import os
import json
if 'CODES' not in sys.path:
    sys.path.append('CODES')
    
from ImageHandler import ImageHandler
import utils_runup 

configPath = "config.json"
print(os.getcwd())
sample_img_path = 'DATA/1724841000.Wed.Aug.28_10_30_00.GMT.2024.CACO03.c2.snap.jpg'

#yaml_filename = "filename to save dictionary of coordinates E,N,Z,U,V - end in .yaml"
yaml_filename = 'CACO03_c2_timestack_20240827.yaml'
#pix_filename = "filename for saving .pix file - end in .pix"
pix_filename = 'CACO03_c2_timestack_20240827.pix'

img_handle = ImageHandler(imagePath = sample_img_path, configPath = configPath)

print(f'sample image path= {sample_img_path}')


img_handle.read_metadata()
metadata = img_handle.metadata
config = img_handle.config

productsPath = config.get("productsPath", {})
pixsaveDir = config.get("pixsaveDir", {})

utils_runup.define_UV_transects(productsPath, 
                    metadata['intrinsics'], 
                    metadata['extrinsics'], 
                    pixsaveDir=pixsaveDir, 
                    yaml_filename = yaml_filename, 
                    pix_filename = pix_filename)         



## Rectify image products

In [ ]:
import sys

if 'CODES' not in sys.path:
    sys.path.append('CODES')

from ImageHandler import ImageDatastore

# Set up datastore for all files in imageDir
datastore = ImageDatastore(configPath = 'config.json')
datastore.image_stats()
# Only keep image products
datastore.keep_images_by_type(['timex','transect', 'snap','var', 'bright', 'dark'])
datastore.image_stats()

# Load all images to datastore
#datastore.initialize_image_handlers()

# Rectify each image onto grid provided in productsPath
datastore.apply_to_all_images('rectify_image')

# Save rectified image to netcdf
datastore.apply_to_all_images('save_to_netcdf')

# Create video of brightest oblique images - can also add start_time and end_time
datastore.create_video(image_type = 'timex', frame_rate = 12, camera = '2')#, end_time=datetime(2024, 8,17,00,00,00, tzinfo=timezone.utc))

# Rectify images, merge camers and save full image with text overlap of site, camera, time. Does not store rectified/merged images.
datastore.merge_images_fast(make_plot=True, save_file=True)


### Example plot of rectified image

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
ds = xr.open_dataset("DATA/DATA/results/netcdf/1729792800.Thu.Oct.24_18_00_00.GMT.2024.CACO03.c1.bright.nc")
ds
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
fig.suptitle(f"Rectified Image - {ds.attrs['name']}", fontsize=14)

# First subplot: pcolor
ax1 = axes[0]
pcolor_plot = ax1.pcolor(ds['eastings_grid'], ds['northings_grid'], ds['Ir'], shading='auto')
ax1.scatter(ds.attrs['origin_easting'], ds.attrs['origin_northing'], c='r')
ax1.set_xlabel("Eastings")
ax1.set_ylabel("Northings")

# Second subplot: imshow
ax2 = axes[1]
im = ax2.imshow(
    ds['Ir'],
    extent=[ds['localX_grid'][0, 0], ds['localX_grid'][-1, -1], ds['localY_grid'][0, 0], ds['localY_grid'][-1, -1]],
    origin='lower'
)
ax2.set_xlabel("utm X")
ax2.set_ylabel("Local Y")

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout for title


### Rectifying single image

In [ ]:

import sys

if 'CODES' not in sys.path:
    sys.path.append('CODES')

from ImageHandler import ImageHandler

imagePath = 'path/to/image'
image_handler = ImageHandler(imagePath = imagePath, configPath = 'config.json')
image_handler.rectify_image()
image_handler.save_to_netcdf()

In [ ]:
from PIL import Image, TiffTags

# Open the TIFF image
img = Image.open('DATA/DATA/images/1739968200.Wed.Feb.19_12_30_00.GMT.2025.CACO04.c1.ras.tiff')

# Extract EXIF data
exif_data = {TiffTags.TAGS[key]: img.tag[key] for key in img.tag_v2}

Print EXIF data
for tag, value in exif_data.items():
    print(f"{tag}: {value}")

In [ ]:
import exifread

# Open image file for reading (binary mode)
with open('DATA/DATA/1738526400.Sun.Feb.02_20_00_00.GMT.2025.CACO04.c1.ras_transect30.png', 'rb') as f:
# Return Exif tags
    tags = exifread.process_file(f)

# Print the tag/value pairs
for tag in tags.keys():
    if tag not in ('JPEGThumbnail', 'TIFFThumbnail', 'Filename', 'EXIF MakerNote'):
        print(f"Key: {tag}, value {tags[tag]}")

## Extract runup

In [ ]:

import sys
import json

if 'CODES' not in sys.path:
    sys.path.append('CODES')

from ImageHandler import ImageDatastore

with open('config.json', "r") as f:
    config = json.load(f)
    

# # Split .tiff files from i2RGUS into individual transects
# if config.get('split_tiff', False) is True:
#     import utils_runup
#     utils_runup.split_tiff(config)

# # Set up datastore for all files in imageDir
datastore = ImageDatastore(configPath = config)

# Remove image products -> keep transects
datastore.remove_images_by_type(['snap', 'timex', 'var', 'bright', 'dark'])
datastore.image_stats()


# Get runup from segformer model (segformerCodeDir/segformerModel), rectify onto DEM (demPath) and save to netcdf
datastore.get_runup_from_timestacks(segformer_flag=True, save_flag=True)


Total images: 5736
Image types and their counts:
  ras_transect20: 767
  ras_transect100: 1207
  ras_transect70: 1664
  ras_transect50: 422
  ras_transect150: 930
  ras_transect30: 484
  ras_transect40: 262
Metadata missing. Writing metadata...
Metadata missing. Writing metadata...
Filtering out
Moved filtered image (blurry) to DATA/DATA/images/filtered/1741611600.Mon.Mar.10_13_00_00.GMT.2025.CACO04.c1.ras_transect70.png
Filtering out
Moved filtered image (blurry) to DATA/DATA/images/filtered/1742315400.Tue.Mar.18_16_30_00.GMT.2025.CACO04.c1.ras_transect50.png
Metadata missing. Writing metadata...
Filtering out
Moved filtered image (blurry) to DATA/DATA/images/filtered/1732212000.Thu.Nov.21_18_00_00.GMT.2024.CACO03.c1.ras_transect50.png
Filtering out
Moved filtered image (blurry) to DATA/DATA/images/filtered/1741554000.Sun.Mar.09_21_00_00.GMT.2025.CACO04.c2.ras_transect150.png
Filtering out
Moved filtered image (blurry) to DATA/DATA/images/filtered/1738672200.Tue.Feb.04_12_30_00.GMT.20

/home/cvolpano/runup-Chelsea/CODES/utils_runup.py:735: RuntimeWarning: divide by zero encountered in scalar divide
  RUstats["Ts"] = t_sec[-1] / len(peaks)
/home/cvolpano/runup-Chelsea/CODES/utils_runup.py:736: RuntimeWarning: divide by zero encountered in scalar divide
  RUstats["Tr"] = t_sec[-1] / len(peaks)
/home/cvolpano/runup-Chelsea/CODES/utils_runup.py:748: RuntimeWarning: divide by zero encountered in scalar divide
  Tp = 1 / fp


Extracting runup
grayscaleDir in get_runup_from_segformer DATA/DATA/results/grayscale
Issue getting runup for 1736973000.Wed.Jan.15_20_30_00.GMT.2025.CACO03.c1.ras_transect100: array of sample points is empty
Getting runup
Issues getting runup from 1736973000.Wed.Jan.15_20_30_00.GMT.2025.CACO03.c1.ras_transect100: ValueError: No SegFormer runup for 1736973000.Wed.Jan.15_20_30_00.GMT.2025.CACO03.c1.ras_transect100 (missing .npz or extraction failed).
Processing: DATA/DATA/images/1741611600.Mon.Mar.10_13_00_00.GMT.2025.CACO04.c1.ras_transect70.png.
Skipping filtered image: 1741611600.Mon.Mar.10_13_00_00.GMT.2025.CACO04.c1.ras_transect70
Processing: DATA/DATA/images/1742315400.Tue.Mar.18_16_30_00.GMT.2025.CACO04.c1.ras_transect50.png.
Skipping filtered image: 1742315400.Tue.Mar.18_16_30_00.GMT.2025.CACO04.c1.ras_transect50
Processing: DATA/DATA/images/1739219400.Mon.Feb.10_20_30_00.GMT.2025.CACO04.c1.ras_transect70.png.
Running Segformer for 1739219400.Mon.Feb.10_20_30_00.GMT.2025.CACO04.

Traceback (most recent call last):
  File "/home/cvolpano/runup-Chelsea/CODES/ImageHandler.py", line 485, in get_runup_from_segformer
    Ri, _, _, _, _, _ = utils_segformer.get_SegGym_runup_pixel_timeseries(av_prob_stack, rundown_val, runup_val, buffer = [0,2])
  File "/home/cvolpano/runup-Chelsea/CODES/utils_segformer.py", line 37, in get_SegGym_runup_pixel_timeseries
    Ri_0 = maskExtractionCV(softmax <= 0, buffer)
  File "/home/cvolpano/runup-Chelsea/CODES/utils_segformer.py", line 137, in maskExtractionCV
    Ri = np.interp(x, x[id], Ri[id])
  File "/home/cvolpano/miniforge3/envs/shorescan_core/lib/python3.10/site-packages/numpy/lib/function_base.py", line 1599, in interp
    return interp_func(x, xp, fp, left, right)
ValueError: array of sample points is empty
Traceback (most recent call last):
  File "/home/cvolpano/runup-Chelsea/CODES/ImageHandler.py", line 2302, in get_runup_from_timestacks
    img_handler.compute_runup(dem = dem)
  File "/home/cvolpano/runup-Chelsea/CODES/Im

Extracting runup
grayscaleDir in get_runup_from_segformer DATA/DATA/results/grayscale
Saved overlay image: DATA/DATA/results/runup/overlay_1739219400.Mon.Feb.10_20_30_00.GMT.2025.CACO04.c1.ras_transect70.jpg
Getting runup
getting tide
Water level data saved to DATA/DATA/results/twl/Boston,_MA_1548_2025-02-10.json
computing runup stats
Saving
Saving runup
All data saved to DATA/DATA/results/netcdf/1739219400.Mon.Feb.10_20_30_00.GMT.2025.CACO04.c1.ras_transect70_runup.nc
Processing: DATA/DATA/images/1732212000.Thu.Nov.21_18_00_00.GMT.2024.CACO03.c1.ras_transect50.png.
Skipping filtered image: 1732212000.Thu.Nov.21_18_00_00.GMT.2024.CACO03.c1.ras_transect50
Processing: DATA/DATA/images/1741554000.Sun.Mar.09_21_00_00.GMT.2025.CACO04.c2.ras_transect150.png.
Skipping filtered image: 1741554000.Sun.Mar.09_21_00_00.GMT.2025.CACO04.c2.ras_transect150
Processing: DATA/DATA/images/1738672200.Tue.Feb.04_12_30_00.GMT.2025.CACO04.c1.ras_transect70.png.
Skipping filtered image: 1738672200.Tue.Feb.04_

/home/cvolpano/runup-Chelsea/CODES/utils_runup.py:735: RuntimeWarning: divide by zero encountered in scalar divide
  RUstats["Ts"] = t_sec[-1] / len(peaks)
/home/cvolpano/runup-Chelsea/CODES/utils_runup.py:736: RuntimeWarning: divide by zero encountered in scalar divide
  RUstats["Tr"] = t_sec[-1] / len(peaks)
/home/cvolpano/runup-Chelsea/CODES/utils_runup.py:748: RuntimeWarning: divide by zero encountered in scalar divide
  Tp = 1 / fp


... (3,013 outputs left out)

Traceback (most recent call last):
  File "/home/cvolpano/runup-Chelsea/CODES/ImageHandler.py", line 485, in get_runup_from_segformer
    Ri, _, _, _, _, _ = utils_segformer.get_SegGym_runup_pixel_timeseries(av_prob_stack, rundown_val, runup_val, buffer = [0,2])
  File "/home/cvolpano/runup-Chelsea/CODES/utils_segformer.py", line 37, in get_SegGym_runup_pixel_timeseries
    Ri_0 = maskExtractionCV(softmax <= 0, buffer)
  File "/home/cvolpano/runup-Chelsea/CODES/utils_segformer.py", line 137, in maskExtractionCV
    Ri = np.interp(x, x[id], Ri[id])
  File "/home/cvolpano/miniforge3/envs/shorescan_core/lib/python3.10/site-packages/numpy/lib/function_base.py", line 1599, in interp
    return interp_func(x, xp, fp, left, right)
ValueError: array of sample points is empty
Traceback (most recent call last):
  File "/home/cvolpano/runup-Chelsea/CODES/ImageHandler.py", line 2302, in get_runup_from_timestacks
    img_handler.compute_runup(dem = dem)
  File "/home/cvolpano/runup-Chelsea/CODES/Im

Extracting runup
grayscaleDir in get_runup_from_segformer DATA/DATA/results/grayscale
Saved overlay image: DATA/DATA/results/runup/overlay_1731249000.Sun.Nov.10_14_30_00.GMT.2024.CACO03.c1.ras_transect70.jpg
Getting runup
getting tide
computing runup stats
Saving
Saving runup
All data saved to DATA/DATA/results/netcdf/1731249000.Sun.Nov.10_14_30_00.GMT.2024.CACO03.c1.ras_transect70_runup.nc
Processing: DATA/DATA/images/1738866600.Thu.Feb.06_18_30_00.GMT.2025.CACO04.c1.ras_transect70.png.
Skipping filtered image: 1738866600.Thu.Feb.06_18_30_00.GMT.2025.CACO04.c1.ras_transect70
Processing: DATA/DATA/images/1742144400.Sun.Mar.16_17_00_00.GMT.2025.CACO04.c2.ras_transect100.png.
Running Segformer for 1742144400.Sun.Mar.16_17_00_00.GMT.2025.CACO04.c2.ras_transect100, look for grayscale image
Extracting runup
grayscaleDir in get_runup_from_segformer DATA/DATA/results/grayscale
Saved overlay image: DATA/DATA/results/runup/overlay_1742144400.Sun.Mar.16_17_00_00.GMT.2025.CACO04.c2.ras_transect10

/home/cvolpano/runup-Chelsea/CODES/utils_runup.py:735: RuntimeWarning: divide by zero encountered in scalar divide
  RUstats["Ts"] = t_sec[-1] / len(peaks)
/home/cvolpano/runup-Chelsea/CODES/utils_runup.py:736: RuntimeWarning: divide by zero encountered in scalar divide
  RUstats["Tr"] = t_sec[-1] / len(peaks)
/home/cvolpano/runup-Chelsea/CODES/utils_runup.py:748: RuntimeWarning: divide by zero encountered in scalar divide
  Tp = 1 / fp


All data saved to DATA/DATA/results/netcdf/1740502800.Tue.Feb.25_17_00_00.GMT.2025.CACO04.c2.ras_transect100_runup.nc
Processing: DATA/DATA/images/1742481000.Thu.Mar.20_14_30_00.GMT.2025.CACO04.c1.ras_transect40.png.
Skipping filtered image: 1742481000.Thu.Mar.20_14_30_00.GMT.2025.CACO04.c1.ras_transect40
Processing: DATA/DATA/images/1742486400.Thu.Mar.20_16_00_00.GMT.2025.CACO04.c1.ras_transect70.png.
Skipping filtered image: 1742486400.Thu.Mar.20_16_00_00.GMT.2025.CACO04.c1.ras_transect70
Processing: DATA/DATA/images/1741959000.Fri.Mar.14_13_30_00.GMT.2025.CACO04.c1.ras_transect40.png.
Skipping filtered image: 1741959000.Fri.Mar.14_13_30_00.GMT.2025.CACO04.c1.ras_transect40
Processing: DATA/DATA/images/1742583600.Fri.Mar.21_19_00_00.GMT.2025.CACO04.c2.ras_transect100.png.
Running Segformer for 1742583600.Fri.Mar.21_19_00_00.GMT.2025.CACO04.c2.ras_transect100, look for grayscale image
Extracting runup
grayscaleDir in get_runup_from_segformer DATA/DATA/results/grayscale
Saved overlay i

In [ ]:
# find the earliest available date for theTWL forecast.
import requests
from datetime import date, timedelta

BASE = "https://coastal.er.usgs.gov/hurricanes/research/twlviewer/api/"
region_id = requests.get(f"{BASE}regions").json()[0]["id"]   # twl_region = 1 (1-based)

def has_forecast(d, time="00:00:00"):
    """True if the API returns at least one forecast for date d (time=None drops the filter)."""
    q = f"date={d}" + (f"&time={time}" if time else "")
    return len(requests.get(f"{BASE}regions/{region_id}/forecasts?{q}").json()) > 0

def earliest_forecast(lo, hi, time="00:00:00"):
    """Binary search for the first date in [lo, hi] with a forecast; hi must have one."""
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if has_forecast(mid, time):
            hi = mid              # forecast exists -> boundary is at or before mid
        else:
            lo = mid + timedelta(days=1)
    return lo

lo, hi = date(2024, 10, 8), date(2024, 10, 16)   # hi: a date you know works
assert has_forecast(hi), "Pick an upper date that has a forecast"
first = earliest_forecast(lo, hi)
print("Earliest forecast (time=00:00:00):", first)

# Check the days around the result for gaps in the record
for d in (first + timedelta(days=k) for k in range(-3, 4)):
    print(d, has_forecast(d), has_forecast(d, time=None))

### Runup from single image

In [ ]:
In [1]: %matplotlib inline

In [ ]:
import sys
import json
if 'CODES' not in sys.path:
    sys.path.append('CODES')

from ImageHandler import ImageHandler
with open('config.json', "r") as f:
    config = json.load(f)
# Set up image
img_handle = ImageHandler(imagePath="DATA/DATA/images/1727796600.Tue.Oct.01_15_30_00.GMT.2024.CACO03.c2.ras_transect70.png", configPath = config)
# Run segformer model
img_handle.run_segformer_on_timestack()
# # Extract runup
img_handle.get_runup_from_segformer()
# # Project horizontal runup to vertical and get runup stats
img_handle.compute_runup()
# # Save to netcdf
img_handle.save_to_netcdf()

### DEM plots

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()

# Plot the surface
fig.add_trace(go.Surface(z=z[::100], x=X[::100], y=Y[::100], colorscale='earth', opacity=1, showscale=False,
    colorbar=dict(title="Elevation (m)", len=0.75, thickness=20),
    contours=dict(
        x=dict(show=True, color="black"),
        y=dict(show=True, color="black"),
        z=dict(show=True, highlightcolor="black", project_z=True)
    )))

# Scatter plot for extrinsics points (Ray origin)
fig.add_trace(go.Scatter3d(x=[extrinsics['x']], y=[extrinsics['y']], z=[extrinsics['z']], mode='markers', marker=dict(size=5, color='blue', opacity=0.7), name='Extrinsics'))

transect_data = coords_grid['transect_0']
for idx, Points_w_set in enumerate(transect_data['xyz']):
    
    fig.add_trace(go.Scatter3d(
            x=[Points_w_set[0]], 
            y=[Points_w_set[1]], 
            z=[Points_w_set[2]], 
            mode='markers', 
            marker=dict(size=5, opacity=0.7, color='black'),
            name=f'Crossing point'  # Naming each set based on the iteration
        ))
fig.update_layout(
    scene=dict(
        aspectmode='manual',
        aspectratio=dict(x=3, y=1, z=1),
        xaxis=dict(
            title='X-axis',
            range=[extrinsics['x']-100, extrinsics['x']+150]  # Replace min_x and max_x with your desired limits
        ),
        yaxis=dict(
            title='Y-axis',
            range=[extrinsics['y']-150, extrinsics['y']+150]  # Replace min_y and max_y with your desired limits
        ),
        zaxis=dict(
            title='Z-axis',
            range=[-2,22]  # Replace min_x and max_x with your desired limits
        ),
    ),
    title=f'3D Surface at {45} degrees',  # Change 45 to your angle_deg value
    margin=dict(r=10, b=10, l=10, t=40) # Adjust margins as needed
)
fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Create the figure
fig = go.Figure()

# Plot the surface
fig.add_trace(go.Surface(z=z[::100], x=X[::100], y=Y[::100], colorscale='earth', opacity=1, showscale=False,
    colorbar=dict(title="Elevation (m)", len=0.75, thickness=20),
    contours=dict(
        x=dict(show=True, color="black"),
        y=dict(show=True, color="black"),
        z=dict(show=True, highlightcolor="black", project_z=True)
    )))

fig.add_trace(go.Scatter3d(
        x=grid['eastings_shoreline'], 
        y=grid['northings_shoreline'], 
        z=grid['elevation_shoreline'], 
        mode='markers', 
        marker=dict(size=5, opacity=0.7, color = 'black'),
        name=f'Shoreline'  # Naming each set based on the iteration
    ))

fig.add_trace(go.Scatter3d(x=t2['Hrunup_utm'].values[:,0], y=t2['Hrunup_utm'].values[:,1], z=t2['TWL'].values, mode='markers', marker=dict(size=5, opacity=0.7),
        name=f'Transect 2'))
fig.add_trace(go.Scatter3d(x=t3['Hrunup_utm'].values[:,0], y=t3['Hrunup_utm'].values[:,1], z=t3['TWL'].values, mode='markers', marker=dict(size=5, opacity=0.7),
        name=f'Transect 3'))
fig.add_trace(go.Scatter3d(x=t4['Hrunup_utm'].values[:,0], y=t4['Hrunup_utm'].values[:,1], z=t4['TWL'].values, mode='markers', marker=dict(size=5, opacity=0.7),
        name=f'Transect 4'))
fig.add_trace(go.Scatter3d(x=t5['Hrunup_utm'].values[:,0], y=t5['Hrunup_utm'].values[:,1], z=t5['TWL'].values, mode='markers', marker=dict(size=5, opacity=0.7),
        name=f'Transect 5'))
fig.add_trace(go.Scatter3d(x=t6['Hrunup_utm'].values[:,0], y=t6['Hrunup_utm'].values[:,1], z=t6['TWL'].values, mode='markers', marker=dict(size=5, opacity=0.7),
        name=f'Transect 6'))


# Labels and title
fig.update_layout(
    scene=dict(
        aspectmode='manual',
        aspectratio=dict(x=3, y=1, z=1),
        xaxis=dict(
            title='X-axis',
            range=[grid.attrs['origin_easting'], grid.attrs['origin_easting']+100]  # Replace min_x and max_x with your desired limits
        ),
        yaxis=dict(
            title='Y-axis',
            range=[grid.attrs['origin_northing']-50, grid.attrs['origin_northing']+80]  # Replace min_y and max_y with your desired limits
        ),
        zaxis=dict(
            title='Z-axis',
            range=[-2,6]  # Replace min_x and max_x with your desired limits
        ),
    ),
    title=f'3D Surface at {45} degrees',  # Change 45 to your angle_deg value
    margin=dict(r=10, b=10, l=10, t=40) # Adjust margins as needed
)


# Show plot
fig.show()


## Extract shorelines

In [ ]:
import sys


if 'CODES' not in sys.path:
    sys.path.append('CODES')

from ImageHandler import ImageDatastore


# Set up datastore for all files in imageDir
datastore = ImageDatastore(configPath = 'config.json')
# Only run on bright images
datastore.keep_images_by_type(['bright'])
datastore.image_stats()

datastore.apply_to_all_images('filter_image')
# Rectify each image onto grid provided in productsPath
datastore.apply_to_all_images('rectify_image')
# Extract shoreline from oblique image and rectify to DEM (demPath) - will processes matching bright and timex
datastore.process_shorelines(make_plots=False, save_flag = True)


### Shoreline from single image

In [ ]:
import sys

if 'CODES' not in sys.path:
    sys.path.append('CODES')
from ImageHandler import ImageHandler

# Set up image
img_handle = ImageHandler(configPath = 'config.json', imagePath = 'DATA/DATA/images/Bright/temp/1739905200.Tue.Feb.18_19_00_00.GMT.2025.CACO04.c2.bright.jpg')
# Rectify image
img_handle.rectify_image(dem_flag = True)
# Extract brightest max shoreline
img_handle.process_bright()
# Project shoreline onto DEM (demPath)
img_handle.rectify_shoreline()
# Save shoreline to netcdf
img_handle.save_to_netcdf()